# CoChem-GEOM - Authentic Rotational Spectroscopy & Structural Fitting Pipeline

This notebook invokes the genuine `cochem_geom` package APIs directly:
- Experimental $B_0$ isotopologue loading (`IsotopologueObservation`, Piccardo et al. 2015 $\mathrm{H_2O}$ benchmark packet)
- Weighted nonlinear least-squares $r_0$ structural fitting (`fit_c2v_triatomic_r0`, $w_i = 1/\sigma_i^2$)
- Honest Kraitchman $r_s$ coordinate reconstruction (`reconstruct_c2v_triatomic_rs`, reporting negative $r_s^2$ imaginary axes without clipping)
- Structural comparison reporting (`render_structural_comparison_latex`)

All quantities are computed by authentic `cochem_geom` / `scipy` / `numpy` routines; fabricated SCF placeholders and synthetic fallbacks are strictly prohibited.

In [ ]:
import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = repo_root / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from cochem_geom import (
    IsotopologueObservation,
    ROTATIONAL_CONSTANT_FACTOR_MHZ_U_A2,
    __version__,
    fit_c2v_triatomic_r0,
    get_exact_isotope_mass,
    reconstruct_c2v_triatomic_rs,
    render_structural_comparison_latex,
)

print(f"[CoChem-GEOM v{__version__}] CODATA factor = {ROTATIONAL_CONSTANT_FACTOR_MHZ_U_A2:.6f} MHz u A^2")


## 1. Weighted $r_0$ Structural Fit & Honest Kraitchman $r_s$ Analysis ($\mathrm{H_2O}$ Benchmark Packet)

In [ ]:
packet_path = repo_root / "benchmarks" / "h2o_t_geom_01" / "h2o_experimental_b0_piccardo2015.json"
if not packet_path.is_file():
    raise FileNotFoundError(f"Missing required experimental benchmark packet: {packet_path}")

packet = json.loads(packet_path.read_text(encoding="utf-8"))
isotopologues = [IsotopologueObservation.model_validate(item) for item in packet["isotopologues"]]

r0_fit = fit_c2v_triatomic_r0(isotopologues)
print(f"Converged: {r0_fit.converged} (n_obs={r0_fit.n_observations}, dof={r0_fit.degrees_of_freedom})")
print(f"r0(O-H)       = {r0_fit.fitted_values['r(O-H)']:.6f} A")
print(f"angle0(H-O-H) = {r0_fit.fitted_values['angle(H-O-H)']:.4f} deg")

iso_map = {iso.species_id: iso for iso in isotopologues}
rs_result = reconstruct_c2v_triatomic_rs(
    parent_abc_mhz=iso_map["H2O"].rotational_constants_mhz,
    parent_unc_mhz=iso_map["H2O"].uncertainties_mhz,
    central_sub_abc_mhz=iso_map["H2(18O)"].rotational_constants_mhz,
    central_sub_unc_mhz=iso_map["H2(18O)"].uncertainties_mhz,
    terminal_sub_abc_mhz=iso_map["HDO"].rotational_constants_mhz,
    terminal_sub_unc_mhz=iso_map["HDO"].uncertainties_mhz,
    mass_central_u=get_exact_isotope_mass("16O"),
    mass_central_iso_u=get_exact_isotope_mass("18O"),
    mass_terminal_u=get_exact_isotope_mass("1H"),
    mass_terminal_iso_u=get_exact_isotope_mass("2H"),
)
for atom_label, atom_res in rs_result.atom_results.items():
    axis_summary = {
        axis: ("IMAGINARY" if rec.imaginary_axis else f"{rec.abs_coord_angstrom:.5f} A")
        for axis, rec in atom_res.axes.items()
    }
    print(f"Kraitchman r_s [{atom_label}]: {axis_summary}")


## 2. Verified Structural Comparison Artifact & LaTeX Table Rendering

In [ ]:
comparison_path = repo_root / "benchmarks" / "h2o_t_geom_01" / "h2o_t_geom_01_comparison.json"
if not comparison_path.is_file():
    raise FileNotFoundError(f"Missing verified comparison payload: {comparison_path}")

comparison_payload = json.loads(comparison_path.read_text(encoding="utf-8"))
latex_table = render_structural_comparison_latex(comparison_payload)
print(latex_table)
